In [ ]:
import pandas as pd
import geopandas as gpd
import openpyxl
from shapely.geometry import Point
import json
import folium

geral = pd.read_excel('Geral.xlsx', sheet_name='Tabela Geral')
# geral['Ano'] = geral['Ano'].astype(int)
# geral['Processo SEI'] = geral['Processo SEI'].astype(int)

geral.rename(columns={'Coordenada X': 'lon', 'Coordenada Y': 'lat', 'Unidade': 'Secretaria', 'Equipamento 1': 'Tipo Equipamento', 'Equipamento 2': 'Sub Tipo Equipamento'}, inplace=True)
dadosgeo = geral
dadosgeo['lon'] = pd.to_numeric(geral['lon'], errors='coerce')
dadosgeo['lat'] = pd.to_numeric(geral['lat'], errors='coerce')
dadosgeo = geral.dropna(subset=['lon', 'lat'])

geometry = [Point(xy) for xy in zip(dadosgeo["lon"], dadosgeo["lat"])]
pontos = gpd.GeoDataFrame(dadosgeo, geometry=geometry)
#pontos.drop(columns=['lon', 'lat'], inplace=True)
pontos.set_crs(epsg=4326, inplace=True)

In [ ]:
#Importa a s geometrias das subprefeituras e distritos, mas apenas as colunas necessárias nome e polígono
subs = gpd.read_file('subprefeituras.geojson', encoding='utf-8')[['nm_subprefeitura','geometry']]
distritos = gpd.read_file('EPSG4326-DistritosSP.geojson', encoding='utf-8')[['nm_distrito_municipal', 'geometry']]


In [ ]:
#iloc aqui isola a sub
print(type(subs.iloc[0].geometry))
print(subs.iloc[0].nm_subprefeitura)
subs.iloc[0].geometry

In [ ]:
pontos.geometry

In [ ]:
subs.iloc[i].geometry.intersects(pontos.geometry).value_counts().get(True, 0)

In [ ]:
soma =0
for i in range(len(subs)):
    a = subs.iloc[i].geometry.intersects(pontos.geometry).value_counts().get(True, 0)
    soma = soma + a
    print(a,soma)

In [ ]:
a = gpd.sjoin(pontos, subs, how="left", predicate="within")
b = gpd.sjoin(pontos, distritos, how="left", predicate="within")
# a.drop(columns=['index_right'], inplace=True)
# b.drop(columns=['index_right'], inplace=True)
# pontos =  a.nm_subprefeitura)
# pontos.add(b.nm_distrito_municipal)
# pontos